# Evaluation of Translation Models

## 1 Overview

## 2 Importing Libraries

In [1]:
# importing libraries
import gc
import random
import time

from pathlib import Path

import pandas as pd
import torch

from datasets import load_dataset
from jiwer import cer, wer
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoProcessor,
    AutoTokenizer,
    M2M100ForConditionalGeneration,
    M2M100Tokenizer,
    SeamlessM4TForTextToText
)

c:\Users\Subathra\OneDrive\Desktop\cm3020_Final_Year_project\CM3020_Final_Year_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3 Evaluation Settings and Candidate Models

In [2]:
# result reproducible
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
# use gpu when it is available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = (
    torch.float16
    if torch.cuda.is_available()
    else torch.float32
)

print("Device:", device)

Device: cuda


In [4]:
# evaluation sizes
VALIDATION_SIZE = 100
TEST_SIZE = 100

In [5]:
# output folder to store results
output_folder = Path("outputs/translation")
output_folder.mkdir(parents=True, exist_ok=True)

In [6]:
# Candidate translation models
models = {
    "NLLB-200": {
        "model_id": "facebook/nllb-200-distilled-600M",
        "batch_size": 8
    },

    "M2M100": {
        "model_id": "facebook/m2m100_418M",
        "batch_size": 8
    },

    "SeamlessM4T": {
        "model_id": "facebook/hf-seamless-m4t-medium",
        "batch_size": 2
    }
}

In [7]:
# target languages
languages = ["Malay", "Chinese", "Tamil"]

In [8]:
print("Models:", len(models))
print("Languages:", languages)

Models: 3
Languages: ['Malay', 'Chinese', 'Tamil']


In [9]:
model_information = []

for model_name, details in models.items():
    model_information.append({
        "Model": model_name,
        "Model ID": details["model_id"]
    })

model_information_df = pd.DataFrame(model_information)
model_information_df

,Model,Model ID
0,NLLB-200,facebook/nllb-200-distilled-600M
1,M2M100,facebook/m2m100_418M
2,SeamlessM4T,facebook/hf-seamless-m4t-medium


## 4 Load OPUS-100 Dataset

In [10]:
DATASET_ID = "Helsinki-NLP/opus-100"

In [11]:
# language pairs in OPUS-100
dataset_configs = {
    "Malay": {
        "config": "en-ms",
        "target": "ms"
    },
    "Chinese": {
        "config": "en-zh",
        "target": "zh"
    },
    "Tamil": {
        "config": "en-ta",
        "target": "ta"
    }
}

In [12]:
# load validation and test datasets
validation_datasets = {}
testing_datasets = {}

for language, details in dataset_configs.items():
    validation_datasets[language] = load_dataset(
        DATASET_ID,
        details["config"],
        split="validation"
    )

    testing_datasets[language] = load_dataset(
        DATASET_ID,
        details["config"],
        split="test"
    )

In [13]:
# select reproducible samples
validation_indices = {}
testing_indices = {}

for language in languages:
    validation_indices[language] = random.Random(SEED).sample(
        range(len(validation_datasets[language])),
        VALIDATION_SIZE
    )

    testing_indices[language] = random.Random(SEED + 1).sample(
        range(len(testing_datasets[language])),
        TEST_SIZE
    )

In [14]:
# prepare English source and reference translations
validation_data = {}
testing_data = {}

for language in languages:
    target = dataset_configs[language]["target"]

    validation_data[language] = {
        "source": [
            validation_datasets[language][index]["translation"]["en"]
            for index in validation_indices[language]
        ],

        "reference": [
            validation_datasets[language][index]["translation"][target]
            for index in validation_indices[language]
        ]
    }

    testing_data[language] = {
        "source": [
            testing_datasets[language][index]["translation"]["en"]
            for index in testing_indices[language]
        ],

        "reference": [
            testing_datasets[language][index]["translation"][target]
            for index in testing_indices[language]
        ]
    }

In [15]:
print("Validation samples")

for language in languages:
    print(
        language,
        len(validation_data[language]["source"])
    )

print("\nTest samples")

for language in languages:
    print(
        language,
        len(testing_data[language]["source"])
    )

Validation samples
Malay 100
Chinese 100
Tamil 100

Test samples
Malay 100
Chinese 100
Tamil 100


In [16]:
for language in languages:
    print(f"\n{language}")
    print(
        "English:",
        validation_data[language]["source"][0]
    )
    print(
        "Reference:",
        validation_data[language]["reference"][0]
    )


Malay
English: But I know for sure that I love you..
Reference: Tapi saya tahu yang pastinya saya cintakan kamu..

Chinese
English: In the developed countries, we have been witnessing an increase in the proportion of older people relative to the population as a whole for a number of years, while at the same time it is seen that our older people, fortunately, live increasingly longer.
Reference: 在发达国家中，一些年来，老年人在整个人口中的相对比例在增加，同时值得庆幸的是，老年人的寿命越来越长。

Tamil
English: Warning
Reference: சரம்


## 5 Language Code Alignment

In [17]:
# language codes used by each model
language_codes = {
    "Malay": {
        "NLLB-200": "zsm_Latn",
        "M2M100": "ms",
        "SeamlessM4T": "zsm"
    },

    "Chinese": {
        "NLLB-200": "zho_Hans",
        "M2M100": "zh",
        "SeamlessM4T": "cmn"
    },

    "Tamil": {
        "NLLB-200": "tam_Taml",
        "M2M100": "ta",
        "SeamlessM4T": "tam"
    }
}

## 6 Evaluation Method

In [18]:
def calculate_error_rates(predictions, references):
    character_error = cer(
        references,
        predictions
    )

    word_error = wer(
        references,
        predictions
    )

    return character_error, word_error

In [19]:
def create_batches(data, batch_size):
    for start in range(0, len(data), batch_size):
        yield data[start:start + batch_size]

In [20]:
def evaluate_predictions(
    model_name,
    predictions,
    data,
    inference_time
):
    results = []

    for language in languages:
        character_error, word_error = calculate_error_rates(
            predictions[language],
            data[language]["reference"]
        )

        sample_count = len(
            data[language]["source"]
        )

        results.append({
            "model": model_name,
            "language": language,
            "CER": character_error,
            "WER": word_error,
            "inference_time":
                inference_time[language],
            "ms_per_sentence":
                inference_time[language]
                / sample_count
                * 1000
        })

    return pd.DataFrame(results)

## 7 Evaluating Candidate Models on Validation Data

In [21]:
def evaluate_nllb(data):
    model_id = models["NLLB-200"]["model_id"]
    batch_size = models["NLLB-200"]["batch_size"]

    start = time.perf_counter()

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        src_lang="eng_Latn"
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_id,
        torch_dtype=dtype
    ).to(device)

    model.eval()

    loading_time = time.perf_counter() - start

    predictions = {}
    inference_times = {}

    for language in languages:
        translated = []

        start = time.perf_counter()

        target_id = tokenizer.convert_tokens_to_ids(
            language_codes[language]["NLLB-200"]
        )

        for batch in create_batches(
            data[language]["source"],
            batch_size
        ):
            inputs = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=256
            ).to(device)

            with torch.inference_mode():
                outputs = model.generate(
                    **inputs,
                    forced_bos_token_id=target_id,
                    max_new_tokens=256
                )

            translated.extend(
                tokenizer.batch_decode(
                    outputs,
                    skip_special_tokens=True
                )
            )

        inference_times[language] = time.perf_counter() - start
        predictions[language] = translated

    del model, tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    results = evaluate_predictions(
        "NLLB-200",
        predictions,
        data,
        inference_times
    )

    return results, loading_time, predictions

In [22]:
print("Evaluating NLLB-200...")

nllb_results, nllb_loading_time, nllb_predictions = evaluate_nllb(
    validation_data
)

nllb_results

Evaluating NLLB-200...


`torch_dtype` is deprecated! Use `dtype` instead!


,model,language,CER,WER,inference_time,ms_per_sentence
0,NLLB-200,Malay,0.462567,0.655385,4.282706,42.827060
1,NLLB-200,Chinese,0.683653,1.202312,12.449352,124.493521
2,NLLB-200,Tamil,0.581562,0.856887,6.814270,68.142703


In [23]:
def evaluate_m2m100(data):
    model_id = models["M2M100"]["model_id"]
    batch_size = models["M2M100"]["batch_size"]

    start = time.perf_counter()

    tokenizer = M2M100Tokenizer.from_pretrained(model_id)

    model = M2M100ForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=dtype
    ).to(device)

    model.eval()

    loading_time = time.perf_counter() - start

    predictions = {}
    inference_times = {}

    for language in languages:
        translated = []

        tokenizer.src_lang = "en"

        target_id = tokenizer.get_lang_id(
            language_codes[language]["M2M100"]
        )

        start = time.perf_counter()

        for batch in create_batches(
            data[language]["source"],
            batch_size
        ):
            inputs = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=256
            ).to(device)

            with torch.inference_mode():
                outputs = model.generate(
                    **inputs,
                    forced_bos_token_id=target_id,
                    max_new_tokens=256
                )

            translated.extend(
                tokenizer.batch_decode(
                    outputs,
                    skip_special_tokens=True
                )
            )

        inference_times[language] = time.perf_counter() - start
        predictions[language] = translated

    del model, tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    results = evaluate_predictions(
        "M2M100",
        predictions,
        data,
        inference_times
    )

    return results, loading_time, predictions

In [24]:
print("Evaluating M2M100...")

m2m100_results, m2m100_loading_time, m2m100_predictions = evaluate_m2m100(
    validation_data
)

m2m100_results

Evaluating M2M100...


`torch_dtype` is deprecated! Use `dtype` instead!


,model,language,CER,WER,inference_time,ms_per_sentence
0,M2M100,Malay,0.571462,0.807692,5.732507,57.325072
1,M2M100,Chinese,0.723452,1.236994,18.618748,186.187484
2,M2M100,Tamil,0.736687,0.976744,7.488211,74.882108


In [25]:
def evaluate_seamless(data):
    model_id = models["SeamlessM4T"]["model_id"]
    batch_size = models["SeamlessM4T"]["batch_size"]

    start = time.perf_counter()

    processor = AutoProcessor.from_pretrained(model_id)

    model = SeamlessM4TForTextToText.from_pretrained(
        model_id,
        torch_dtype=dtype
    ).to(device)

    model.eval()

    loading_time = time.perf_counter() - start

    predictions = {}
    inference_times = {}

    for language in languages:
        translated = []

        start = time.perf_counter()

        for batch in create_batches(
            data[language]["source"],
            batch_size
        ):
            inputs = processor(
                text=batch,
                src_lang="eng",
                return_tensors="pt",
                padding=True
            ).to(device)

            with torch.inference_mode():
                outputs = model.generate(
                    **inputs,
                    tgt_lang=language_codes[language]["SeamlessM4T"],
                    max_new_tokens=256
                )

            translated.extend(
                processor.batch_decode(
                    outputs,
                    skip_special_tokens=True
                )
            )

        inference_times[language] = time.perf_counter() - start
        predictions[language] = translated

    del model, processor
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    results = evaluate_predictions(
        "SeamlessM4T",
        predictions,
        data,
        inference_times
    )

    return results, loading_time, predictions

In [26]:
print("Evaluating SeamlessM4T...")

seamless_results, seamless_loading_time, seamless_predictions = evaluate_seamless(
    validation_data
)

seamless_results

Evaluating SeamlessM4T...


,model,language,CER,WER,inference_time,ms_per_sentence
0,SeamlessM4T,Malay,0.513729,0.709231,9.150052,91.500519
1,SeamlessM4T,Chinese,0.740190,2.647399,28.383510,283.835098
2,SeamlessM4T,Tamil,0.642391,0.894454,12.062683,120.626830


In [27]:
validation_results_df = pd.concat(
    [
        nllb_results,
        m2m100_results,
        seamless_results
    ],
    ignore_index=True
)

In [28]:
validation_results_df

,model,language,CER,WER,inference_time,ms_per_sentence
0,NLLB-200,Malay,0.462567,0.655385,4.282706,42.827060
1,NLLB-200,Chinese,0.683653,1.202312,12.449352,124.493521
2,NLLB-200,Tamil,0.581562,0.856887,6.814270,68.142703
3,M2M100,Malay,0.571462,0.807692,5.732507,57.325072
4,M2M100,Chinese,0.723452,1.236994,18.618748,186.187484
5,M2M100,Tamil,0.736687,0.976744,7.488211,74.882108
6,SeamlessM4T,Malay,0.513729,0.709231,9.150052,91.500519
7,SeamlessM4T,Chinese,0.740190,2.647399,28.383510,283.835098
8,SeamlessM4T,Tamil,0.642391,0.894454,12.062683,120.626830


## 8 Comparing and Selecting the Best Model

In [29]:
loading_times = {
    "NLLB-200": nllb_loading_time,
    "M2M100": m2m100_loading_time,
    "SeamlessM4T": seamless_loading_time
}

In [30]:
comparison_results = []

for model_name in models:
    results = validation_results_df[
        validation_results_df["model"] == model_name
    ]

    comparison_results.append({
        "Model": model_name,
        "Average CER": results["CER"].mean(),
        "Average WER": results["WER"].mean(),
        "Loading Time": loading_times[model_name],
        "Inference Time": results["inference_time"].sum(),
        "ms / sentence": results["ms_per_sentence"].mean()
    })

comparison_df = pd.DataFrame(
    comparison_results
)

In [31]:
comparison_df = comparison_df.sort_values(
    [
        "Average CER",
        "Average WER",
        "Inference Time"
    ],
    ascending=[
        True,
        True,
        True
    ]
).reset_index(drop=True)

comparison_df

,Model,Average CER,Average WER,Loading Time,Inference Time,ms / sentence
0,NLLB-200,0.575927,0.904861,5.899866,23.546328,78.487761
1,SeamlessM4T,0.632103,1.417028,12.728459,49.596245,165.320816
2,M2M100,0.677200,1.007144,4.845295,31.839466,106.131555


In [32]:
selected_model_name = comparison_df.loc[
    0,
    "Model"
]

print(
    "Selected model:",
    selected_model_name
)

print(
    "Model ID:",
    models[selected_model_name]["model_id"]
)

Selected model: NLLB-200
Model ID: facebook/nllb-200-distilled-600M


## 9 Final Test Evaluation

In [33]:
if selected_model_name == "NLLB-200":
    test_results_df, test_loading_time, test_predictions = evaluate_nllb(
        testing_data
    )

elif selected_model_name == "M2M100":
    test_results_df, test_loading_time, test_predictions = evaluate_m2m100(
        testing_data
    )

else:
    test_results_df, test_loading_time, test_predictions = evaluate_seamless(
        testing_data
    )

In [34]:
test_results_df[
    [
        "language",
        "CER",
        "WER",
        "inference_time",
        "ms_per_sentence"
    ]
]

,language,CER,WER,inference_time,ms_per_sentence
0,Malay,0.474314,0.664604,4.598361,45.983614
1,Chinese,0.683505,1.152542,8.908170,89.081698
2,Tamil,0.561563,0.839757,5.405057,54.050567


In [35]:
test_summary_df = pd.DataFrame([
    {
        "Selected Model": selected_model_name,
        "Average CER": test_results_df["CER"].mean(),
        "Average WER": test_results_df["WER"].mean(),
        "Loading Time": test_loading_time,
        "Inference Time": test_results_df["inference_time"].sum(),
        "ms / sentence": test_results_df["ms_per_sentence"].mean()
    }
])

test_summary_df

,Selected Model,Average CER,Average WER,Loading Time,Inference Time,ms / sentence
0,NLLB-200,0.573127,0.885634,5.721516,18.911588,63.038626


## 10 Translation Examples

In [36]:
translation_examples = []

for language in languages:
    for index in range(5):
        translation_examples.append({
            "Language": language,
            "English":
                testing_data[language]["source"][index],
            "Reference":
                testing_data[language]["reference"][index],
            "Prediction":
                test_predictions[language][index]
        })

translation_examples_df = pd.DataFrame(
    translation_examples
)

translation_examples_df

,Language,English,Reference,Prediction
0,Malay,It has the fuel capacity to reach Mars orbit.,Ia mempunyai minyak yang mencukupi untuk sampa...,Ia mempunyai kapasiti bahan api untuk mencapai...
1,Malay,I want out of this deal!,Saya nak tarik balik modal!,Saya mahu keluar dari perjanjian ini!
2,Malay,- Yeah.,- Ya -,"- Ya, saya boleh."
3,Malay,"Happy Fat Tuesday, officer.","Selamat hari Fat Tuesday, tuan.","Selamat Hari Selasa Lemak, pegawai."
4,Malay,I wanna pop smoke 'em and secure this building.,Aku mahu baling bom asap dan pastikan bangunan...,Saya mahu membakar mereka dan mengamankan bang...
5,Chinese,And what your husband said... if Columbus had ...,你丈夫说的... 要是哥伦布没发现美洲，我们现在就都是印第安人了,"你丈夫说... 如果哥伦布做了,我们都会是印第安人."
6,Chinese,All four alleged perpetrators were arrested be...,所有4名指称的肇事者均因这一罪行于1993年9月3日至4日之间被捕。,据称这四名罪犯都因犯罪而被捕于1993年9月3日至4日.
7,Chinese,"In Geneva, conference services are sometimes p...",在日内瓦，只要及早事先通知，而且工作负荷允许，有时便利用现有资源提供会议服务。,"在日内瓦,会议服务有时从现有资源中提供,只要有足够的预告,工作量允许."
8,Chinese,Agenda item 22 (k),议程项目22(k),议程第22条 (k)
9,Chinese,Then the last line of subparagraph (g) should ...,那么，(g)分段第一行应改为`…不得因所进行与联合国职务有关的活动…'。”,"然后,应修改第 g 项最后一行以读为""......关于与联合国服务有关的活动""."


## 11 Saving Evaluation Results

In [37]:
validation_results_df.to_csv(
    output_folder
    / "translation_validation_results.csv",
    index=False
)

comparison_df.to_csv(
    output_folder
    / "translation_model_comparison.csv",
    index=False
)

test_results_df.to_csv(
    output_folder
    / "selected_translation_model_test_results.csv",
    index=False
)

test_summary_df.to_csv(
    output_folder
    / "selected_translation_model_test_summary.csv",
    index=False
)

translation_examples_df.to_csv(
    output_folder
    / "translation_examples.csv",
    index=False
)

print(
    "Results saved in:",
    output_folder
)

Results saved in: outputs\translation


## 12 Conclusion